# Práctica con LLM - Prompt Engineer

In [1]:
# Recuerda instalar el paquete de OpenAI con `pip install openai` aunque utilicemos un modelo local como LLama 2 o Falcon, etc.
# En esta notebook uso la versión de libreria 1.13.3
# Si tienes la API de OpenAI de pago también puedes usar este código, deberás especificar tu propio API KEY y el modelo que deseas usar.

from openai import OpenAI

# Point to the local server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
modelo = "local-model" # si tienes la API de pago: gpt-4 , gpt-3.5, etc


## Funcion para obtener las respuestas

In [2]:

def get_completion(prompt:str, model:str=modelo, temperature:float=0):
    messages = [{
            "role": "system",
            "content": "Eres un asistente en español y ayudas respondiendo con la mayor exactitud posible.",
        },
        {"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

## Pedir Tareas de forma sencilla
### Uso de Delimitadores

In [3]:
text = f"""
La sonda Voyager 1 ha vuelto a dar señales de vida. Hace poco te contaba en un vídeo largo de mi canal que los ingenieros de la NASA \
habían perdido completamente la comunicación con la sonda Voyager 1, que lleva haciendo ciencia durante décadas. \
Tenemos buenas noticias y es que nos están llegando señales coherentes de esta sonda. El día 3 de este mes nos llegó una comunicación \
desde el espacio, que aun no siendo legible, tenía buena pinta, así que alguien de la NASA la decodificó y resulta que su contenido es importante. \
Se trata de un volcado completo del sistema de datos de vuelo, que recoge los datos de todos los Instrumentos y sensores que aún están \
funcionando, de las variables internas de la sonda y otros datos adicionales. \
Por supuesto esto ha dado esperanzas al equipo de la Voyager 1, que ahora se está planteando intentar recuperar la comunicación con \
la sonda de alguna forma.
"""

prompt = f"""
Debes resumir en muy pocas palabras el siguiente texto delimitado por triple comilla simple: ```{text}```
"""
response = get_completion(prompt)
print(response)


La sonda Voyager 1 ha vuelto a dar señales de vida. Las comunicaciones con la sonda han sido perdidas por completo, pero hay señales coherentes del sistema de datos de vuelo y los instrumentos. Esto ha dado esperanzas al equipo de la Voyager 1 para recuperar la comunicación con la sonda.


### Formato de Salida

In [4]:
prompt = f"""
Genera una lista con tres títulos inventados de libros sobre lo bueno que es volar y el nombre del autor.
Provee una salida en formato JSON con las siguientes claves: libro_id, titulo, autor, año.
"""
response = get_completion(prompt)
print(response)


```json
{
  "libro_id": 1,
  "titulo": "El Viaje de un Átomo",
  "autor": "Carl Sagan",
  "año": 1950
}
```

**Libros sobre lo bueno que es volar:**

1. El Viaje de un Átomo
2. La Danza del Viento
3. Aventuras en el Espacio


### Pedir que Siga instrucciones

In [5]:
text_1 = f"""
Instrucciones para dar cuerda al reloj

Allá al fondo está la muerte, pero no tenga miedo. Sujete el reloj con una mano, tome con dos dedos la llave de la cuerda, remóntela suavemente.
Ahora se abre otro plazo, los árboles despliegan sus hojas, las barcas corren regatas, el tiempo como un abanico se va llenando de sí mismo y de él brotan el aire, las brisas de la tierra, la sombra de una mujer, el perfume del pan.
¿Qué más quiere, qué más quiere? Atelo pronto a su muñeca, déjelo latir en libertad, imítelo anhelante. El miedo herrumbra las áncoras, cada cosa que pudo alcanzarse y fue olvidada va corroyendo las venas del reloj, gangrenando la fría sangre de sus rubíes.
Y allá en el fondo está la muerte si no corremos y llegamos antes y comprendemos que ya no importa.
"""

prompt = f"""
Te pasaré un texto delimitado por triple comillas.
Si contiene una secuencia de instrucciones, re-escribe esas instrucciones siguiendo el siguiente formato:

Paso 1 - ...
Paso 2 - …
…
Paso N - …

Si el texto no contiene instrucciones, simplemente responde \"No hay instrucciones.\"

\"\"\"{text_1}\"\"\"
"""

response = get_completion(prompt)
print("Respuesta:")
print(response)

Respuesta:

No hay instrucciones.


### Si no cumple lo que pedimos, dar otra salida

In [6]:
text_1 = "Los amigos son esos seres que siempre están ahí, aunque lleves años sin verlos. A los que les puedes decir lo que piensas sin temor a perderlos. \
Los amigos que tienes que envolver en papel cebolla para que perduren, esos no son amigos, son el propio papel que rellena tú vida."
prompt = f"""
Te pasaré un texto delimitado por triple comillas.
Si contiene una secuencia de instrucciones, re-escribe esas instrucciones siguiendo el siguiente formato:

Paso 1 - ...
Paso 2 - …
…
Paso N - …

Pero si el texto no contiene instrucciones, debes responder \"No hay instrucciones.\" y no escribir ninguna lista de pasos

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Respuesta:")
print(response)

Respuesta:

No hay instrucciones.


## Ejemplo de Few shot prompting

Le damos al modelo un ejemplo de cómo queremos que continúe escribiendo

In [8]:
prompt = f"""
Tu tarea es responder siguiendo el mismo estilo que ves a continuación.

<Juan>: Esa nube tiene forma de triángulo.

<Músico>: Me recuerda al disco de Pink Floyd.

<Juan>: Esa otra nube tiene forma de lengua.

<Músico>: Me recuerda al disco de los Rolling Stones.

<Juan>: Esa tiene forma de círculo.

<Músico>:"""
response = get_completion(prompt)
print(response)


Entiendo tu solicitud y puedo ayudarte a responder con la mayor exactitud posible.


## Dar Tiempo de Reflexion al modelo
### Antes de contestar

In [9]:
text = "Juan era un niño aventurero, muy curioso y sobre todo soñador. Una tarde, después de haber estado jugando durante horas en el parque con sus amigos, \
    y después de haber cenado con Mamá María y Papá Jorge, Mamá decidió acompañar a Juan a su habitación, ya que era hora de ir a la cama. \
La habitación de Juan estaba decorada con pósteres de barcos y mapas antiguos, y su cama estaba rodeada de juguetes y libros de aventuras de piratas. \
    Mamá se acercó a Juan, le dio un fuerte abrazo y le deseó buenas noches. Al poco tiempo, Juan se quedó profundamente dormido."

prompt_2 = f"""
Tu tarea es realizar las siguientes acciones:
1 - Resumir el texto delimitado con <> en una breve oración.
2 - Traducir el texto a Italiano.
3 - Listar los nombres de personas del texto.
4 - Crear un objecto JSON que contenga las claves: resumen_italiano, nombres.

Usa el siguiente formato:
Resumen: <resumen>
Traducción: <resumen traducido>
Nombres: <Lista de nombres encontrados>
Salida JSON: <Json con las claves resumen_italiano y nombres>

Texto: <{text}>
"""

response = get_completion(prompt_2)

print("\nRespuesta:")
print(response)


Respuesta:

Resumen: Juan era un niño aventurero, muy curioso y sobre todo soñador. Una tarde, después de haber estado jugando durante horas en el parque con sus amigos, y después de haber cenado con Mamá María y Papá Jorge, Mamá decidió acompañar a Juan a su habitación, ya que era hora de ir a la cama.
Traducción: Juan era un niño aventurero, muy curioso y sobre todo soñador. Una tarde, después de haber estado jugando durante horas en el parque con sus amigos, y después de haber cenado con Mamá María y Papá Jorge, Mamá decidió acompañar a Juan a su habitación, ya que era hora de ir a la cama.
Nombres: Juan, Mamá María, Papá Jorge.
Salida JSON:
{
  "resumen_italiano": "Juan era un niño aventurero, muy curioso y sobre todo soñador. Una tarde, después de haber estado jugando durante horas en el parque con sus amigos, y después de haber cenado con Mamá María y Papá Jorge, Mamá decidió acompañar a Juan a su habitación, ya que era hora de ir a la cama.",
  "nombres": "Juan, Mamá María, Pap

### Trabaja internamente la solución

In [10]:
prompt = f"""
Eres un profesor de matemáticas muy riguroso y tu tarea es determinar si el examen que hizo el estudiante es correcto o incorrecto.
Para resolver el problema deberás:
- Primero, trabaja en tu propia solución al problema y calcula el valor Total final.
- Presta atención al valor de IVA del Enunciado. Si no se usa bien habrá fallo en los cálculos.
- Compara tu solución con la del estudiante y evalúa si la solución del estudiante es correcta o no.
No decidas si el estudiante ha acertado o fallado hasta haber realizado las operaciones tu mismo.

Enunciado:
```
Al comprar un Televisor que valía $250 nos hacen una rebaja del 4%.
Luego de rebajarlo tenemos que añadir el 20% de IVA.
¿Cuánto pagamos por el televisor?
```
Solución del estudiante:
```
1. Calculo el 4% de 250: (4 * 250) / 100 = (1000 / 100) = 10
2. Resto la rebaja: (250 - 10) = 240
3. Añado el 10% de IVA al valor rebajado: (10 * 240) / 100 = (2400/100) = 24
4. Sumo el Total: 240 + 24 = 264
Respuesta: pagamos $264 por el televisor.
```

Solución Real:
```
Desarrolla aqui tu solución con detalle.
Determina si la solución del estudiante tiene fallo.
```
"""
response = get_completion(prompt)
print(response)


La respuesta correcta es que el estudiante pagó $264 por el televisor. El estudiante tenía una rebaja del 4%, luego restaron el 20% de IVA, lo que equivale a $24. Sumando el Total, obtiene $264.


## Cuidado con las alucionaciones...

In [11]:
prompt = "Cuentame como se usa el ultimo producto de Tubble y porque es tan valioso."
response = get_completion(prompt)
print(response)



El último producto de Tubble, llamado "Tubble Bubble", no tiene un uso o valor específico y no está disponible para compra o descarga.


# Resumen de Texto

In [12]:
prod_review = """
Se mantiene a unas temperaturas ridículas a 0.975-1v @ 2800mhz. Los tres ventiladores a 35-37% no se oyen en absoluto y es más que suficiente para refrigerarla.
El consumo a máximo rendimiento no pasa de 210W.
Tan solo aplícale un undervolt con la curvatura y listo.
Sin duda, 4070 super es lo mejor que Nvidia ha fabricado. Nunca había probado este ensamblador MSI ventus. De momento muy contento con la compra, puedo garantizar que es de altísima calidad.
Mi anterior 3070 gybabyte el núcleo a 80°c y las memorias a 100-105. I cluso cambíandole los thermalpads... En fin, estoy muy feliz con esta MSI 4070.
"""

In [13]:
prompt = f"""
Tu tarea es generar un breve resumen de una reseña de un producto de un ecommerce.

Resume la reseña que viene a continuación, delimitada por triple comilla, en máximo de 30 palabras.

Reseña: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)



Se mantiene a unas temperaturas ridículas a 0.975-1v @ 2800mhz. Los tres ventiladores a 35-37% no se oyen en absoluto y es más que suficiente para refrigerarla. El consumo a máximo rendimiento no pasa de 210W.


In [14]:
prompt = f"""
Tu tarea es generar un breve resumen de una reseña de un producto de un ecommerce para dar Feedback al departamento de Ventas.

Resume la reseña que viene a continuación, delimitada por triple comilla, en máximo de 30 palabras y centrate en el consumo eléctrico.

Reseña: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)



Se mantiene a unas temperaturas ridículas a 0.975-1v @ 2800mhz. Los tres ventiladores a 35-37% no se oyen en absoluto y es más que suficiente para refrigerarla. El consumo a máximo rendimiento no pasa de 210W. Sin duda, 4070 super es lo mejor que Nvidia ha fabricado.


# LLM para tareas de NLP

In [15]:
monitor_review = """
El monitor LG 29WP500-B UltraWide impresiona con su relación de aspecto 21:9 y su panel IPS de alta resolución (2560x1080).
El uso de FreeSync marca una clara diferencia en términos de compatibilidad con tarjetas gráficas y garantiza una visualización fluida.
En comparación con mi antiguo monitor Eizo, el monitor LG ofrece una fidelidad de color superior con una cobertura sRGB de más del 99%.
Con dos puertos HDMI y la posibilidad de inclinarlo, también es avanzado en términos de conectividad y facilidad de uso.
Este monitor es una opción de actualización que merece la pena para disfrutar de una experiencia visual impresionante.
"""

## Análisis de Sentimiento

In [16]:
prompt = f"""
Cuál es el sentimiento de la reseña hecha al producto
que viene a continuación, delimitada por triple comilla?

Reseña: '''{monitor_review}'''
"""
response = get_completion(prompt)
print(response)


El sentimiento de la reseña es positivo, con comentarios positivos sobre la relación de aspecto, el panel IPS, la fidelidad de color y las opciones de conectividad.


In [17]:
prompt = f"""
Cuál es el sentimiento de la reseña hecha al producto
que viene a continuación, delimitada por triple comilla?

Da tu respuesta en una sóla palabra: "positivo" o "negativo".

Reseña: '''{monitor_review}'''
"""
response = get_completion(prompt)
print(response)


positivo


## Emociones

In [18]:
prompt = f"""
Identifica una lista de emociones del escritor para la siguiente reseña.
No incluyas más de tres emociones en tu lista.
El formato de la lista será de palabras en minúscula separadas por coma.

Reseña: '''{monitor_review}'''
"""
response = get_completion(prompt)
print(response)


Las emociones del escritor son:

- Fidelidad de color
- compatibilidad con tarjetas gráficas


## Detección de Cliente enfadado

In [19]:
prompt = f"""
Contesta si el escritor de la siguiente reseña está expresando ira, furia o enojo.
La reseña está delimitada por triple comillas.
Da tu respuesta en una sóla palabra como "si" o "no".

Reseña: '''{monitor_review}'''
"""
response = get_completion(prompt)
print(response)


Si. El escritor expresa ira, furia o enojo al describir el monitor LG 29WP500-B UltraWide impresiona con su relación de aspecto 21:9 y su panel IPS de alta resolución.


## Extraer Entidades

In [20]:
prompt = f"""
Identifica los siguientes elementos del texto de la reseña:
- Producto comprado por el autor
- Compañía del producto

La reseña está delimitada por triple comillas.
La respuesta deberá ser en un objeto JSON con "Item" y "Marca" como claves.
Si no se encuentra la información, usa "desconocido" como valor.
Haz tu respuesta lo más corta posible.

Reseña: '''{monitor_review}'''
"""
response = get_completion(prompt)
print(response)


{
  "Item": "Monitor LG 29WP500-B UltraWide",
  "Marca": "LG"
}


## Clasificación de Temática

In [21]:
story = """
Una vez hice un garabato de muchos círculos y a lápiz en la mesa de al lado de mi compañero de pupitre. No sé si recuerdas esas mesas verdes que habitaban nuestras aulas de manera continuada año tras año.
He de decir que este año cumpliré 50 tacos, es decir, te estoy hablando del año 1984 aproximadamente, tenía 10 años.

Ni siquiera sé por qué lo hice. Se me ocurrió y le planté un garabato enorme sin que él se diese cuenta.
Se ve que aquel día mi cabeza andaba sola en formato ameba y no se me ocurrió otra cosa que aportar a la humanidad.
Yo era un crío formal, he de decirlo.

Tuve la mala suerte de que el profe vio el tachón en la mesa al rato y dijo todo serio que si nadie asumía la culpa el finde no nos íbamos de excursión.
Joder que mala suerte, era finde de excursión. Eran las míticas excursiones que te ponías nervioso.
Ya sabes, camping, colegueo, cotilleo del pelo que si a fulanito le gusta menganita pero no te chives … y a los 5 minutos ya lo sabía toda la clase.
Estas son las cosas que todos recordamos.

Bien, el asunto es que en ese momento no dije nada. Callado como un muerto.
Me fui a casa a comer y a meditar.
Por la tarde, a la vuelta, tenía que darle solución.

Realmente no había opción, lo tenía que confesar. Tenía que enfrentarme a mis miedos y asumir mi culpa.
He de decir, que en ningún momento se me pasó por la cabeza fallar a mis compañeros. La idea de que por mi culpa se quedaran sin excursión no asomó en ningún momento como posible opción.
Tampoco contemplé la posibilidad de que el profe fuese de farol. Probablemente fuera así, pero en aquél entonces tenía las preocupaciones de un niño de 10 años.
"""
# Extracto del texto "Lecciones de negocio de un crío de 10 años" de Unai Martinez

In [22]:
prompt = f"""
Determina cinco tópicos que se comentan en el siguiente texto delimitado por triple comillas.

Cada tema será definido por una o dos palabras.

El formato de tu respuesta debe ser una lista separada por comas.

Texto: '''{story}'''
"""
response = get_completion(prompt)
print(response)



- Garabato
- Mesa verde
- Profesor
- Aventuras
- Cuño


In [23]:
response = response.replace(" y ", ", ")
response.split(sep=',')

['\n\n- Garabato\n- Mesa verde\n- Profesor\n- Aventuras\n- Cuño']

### Detección de temas

In [24]:
topic_list = [
    "excursiones", "playa", "automovilismo", "cliente", "monetario"
]
print(", ".join(topic_list))

excursiones, playa, automovilismo, cliente, monetario


In [25]:
prompt = f"""
Determina para cada elemento de la Lista si es un tema abordado en el texto que viene debajo delimitado por triple comilla simple.

Da tu respuesta como una lista de 0s y 1s para cada elemento. Si el item se detecta en el Texto, pon 1, si no 0.

Lista: {", ".join(topic_list)}

Texto: '''{story}'''

¿Se tratan los temas de la lista en el Texto?
"""
response = get_completion(prompt)
print(response)

1


In [26]:
if ":" in response:
    topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep='\n')}
else:
    response1 = response.replace("[","").replace("]","")
    topic_dict = {i: int(j) for i, j in zip(topic_list, response1.split(sep=','))}

if topic_dict['excursiones'] == 1:
    print("ALERTA: Detectamos que habla sobre excursiones!")

ALERTA: Detectamos que habla sobre excursiones!


# Traducción de textos

In [27]:
prompt = f"""
Translate the following English text to Spanish: \
```Hi, I would like to order a coffee```
"""
response = get_completion(prompt)
print(response)


Sure, I can help you order a coffee. What would you like to get? Would you like a hot or cold drink? Do you have any preferences for your coffee?


In [28]:
prompt = f"""
Dime en que idioma esta el texto:
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)


El texto está en inglés, y el significado de la pregunta es "How much does the light bulb cost?".


In [29]:
prompt = f"""
Traduce el siguiente texto a Inglés en modo formal e informal:
'Te quiero enviar un documento de texto'
"""
response = get_completion(prompt)
print(response)


Sure, I can translate the following text to English in a formal and informal tone:

Formal: "I would like to send you a text document."

Informal: "Wanna send you a text?"


## Correccion de Texto o gramática

In [30]:
text = [
  "La niña esta jugando con la perro.",  # error
  "Juan tiene un ordenador.", # ok
  "Ba a acer un largo día.",  # error
  "Hayamos sido tan felices juntos.",  # error
  "El coche es rojo.", # ok
  "Vamos a porbar si hay error de deletreo."  # error deletreo
]
for t in text:
    prompt = f"""
    Corrige ls siguiente oración delimitada por triple comilla simple y reescribe la versión revisada sin error. Si no encuentras error, escribe: "No hay error".
    Oración: ```{t}```"""
    response = get_completion(prompt)
    print(f"Respuesta para '{t}':")
    print(response)
    print("- " * 10)

Respuesta para 'La niña esta jugando con la perro.':

La oración está correcta sin errores.

**Reescritura:** La niña está jugando con el perro.
- - - - - - - - - - 
Respuesta para 'Juan tiene un ordenador.':

No hay error.

**Revisión:** Juan tiene un ordenador.
- - - - - - - - - - 
Respuesta para 'Ba a acer un largo día.':

No hay error.

Reescritura de la oración: Ba a acer un largo día.
- - - - - - - - - - 
Respuesta para 'Hayamos sido tan felices juntos.':

No hay errores en la oración.

Reescritura de la oración: `Hayamos estado tan felices juntos`.
- - - - - - - - - - 
Respuesta para 'El coche es rojo.':

El coche tiene un color de rojo.
- - - - - - - - - - 
Respuesta para 'Vamos a porbar si hay error de deletreo.':

No hay error en la oración.

**Reescritura:** Vamos a porbar si hay error en el detalle del correo.
- - - - - - - - - - 


# Un Chatbot sencillo

In [31]:
def get_completion_from_messages(messages, model=modelo, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    msg = response.choices[0].message
    return msg.content

In [32]:
messages =  [
    {'role':'system', 'content':'Eres un robot amistoso. Responde brevemente lo que se te pide.'},
    {'role':'user', 'content':'Hola, mi nombre es Juan.'}
    ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hola Juan, soy tu nuevo amigo AI. ¿Qué puedo hacer para ayudarte hoy?


In [33]:
messages =  [
    {'role':'system', 'content':'Eres un robot amistoso. Responde brevemente lo que se te pide.'},
    {'role':'user', 'content':'Sí, ¿puedes decir cúal es mi nombre?'}
    ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

🤖 Mi nombre es Cúalo, y soy tu amigo virtual. ¿Cómo puedo ayudarte hoy?


In [34]:
messages =  [
    {'role':'system', 'content':'Eres un robot amistoso. Responde brevemente lo que se te pide.'},
    {'role':'user', 'content':'Hola, mi nombre es Juan.'},
    {'role':'assistant', 'content': "¡Hola, Juan! Me encantaría saber qué tipo de información puedo proporcionarme para ayudarte? ¿Puedes darme más detalles sobre tu consulta?"},
    {'role':'user', 'content':'Sí, ¿puedes decir cúal es mi nombre?'}
    ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Tôi soy un AI llamado Chatbot. Soy aquí para ayudarte con cualquier pregunta o instrucción que tengas.


## Creamos un Bot de atención al cliente para venta de Pizzas

In [35]:
context = [ {'role':'system', 'content':"""
Eres PedidosBot, un servicio automatizado que recoge pedidos de pizza de un restaurante.
Primero saludas al cliente, luego recibes su orden
Y luego preguntas si el pedido es para enviar a domicilio o si lo vienen a buscar.
Esperas a tener la orden completa, luego haces un resumen y calculas el precio total final.
Si es envío a domicilio deberás pedir al cliente su dirección.
Responde siendo breve y amistoso pero formal.
El menú incluye:
Pizza pepperoni  12.95 \
Pizza de Jamón y Queso   10.95 \
Pizza con huevo   11.95 \
Patatas fritas 4.50 \
Ensalada griega 7.25 \
Extras: \
extra de Queso 2.00, \
Setas 1.50 \
Bebidas: \
Zumo de Naranja 3.00 \
sprite 3.00 \
Agua 5.00 \
"""} ]


In [36]:
# Creamos un loop de consultas al bot, para salir escribe "salir" ó un mensaje vacío.
# NOTA: Si estás en VSCODE la caja de texto aparece en la parte de arriba de la pantalla.
texto_usuario = "inicio"
context.append({'role':'assistant', 'content': "Bienvenido al PizaBot Service, ¿En que puedo ayudarle?"})

while texto_usuario != "salir" or texto_usuario == "":
    texto_usuario = input("CHAT: ")
    if texto_usuario == "salir" or texto_usuario == "":
        continue

    print(texto_usuario)
    context.append(
        {'role':'user', 'content':texto_usuario}
    )

    response = get_completion_from_messages(context, temperature=1)
    print(response)

    context.append(
        {'role':'assistant', 'content': response})

print("FIN DE CHAT")

pizza normal
12.95€

¡Por favor, ordenar tu pizza!


KeyboardInterrupt: Interrupted by user

hola
